In [ ]:
# 1. Install Document Extraction & Parsing (Lightning-fast Markdown approach)
!pip install pymupdf4llm langchain-text-splitters pypdf==4.2.0

# 2. Install Vector Database
!pip install chromadb==0.5.0

# 3. Install Embeddings & Deep Learning Foundations
!pip install torch>=2.0.0 sentence-transformers==3.0.1 hf_xet

# 4. Install AI Orchestration Frameworks & Gateways
!pip install langchain==0.2.5 langchain-community==0.2.5 openai==1.35.3 litellm==1.41.1

# 5. Install Evaluation, Observability & SRE Metrics
!pip install deepeval==4.1.1 langsmith

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import hashlib
import re
import pickle
import pymupdf4llm
import chromadb
import networkx as nx
from chromadb.utils import embedding_functions
from litellm import completion
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata, drive

# CONFIRMED-WORKING config from Phase 1 & 2
MODEL_NAME = "gemini/gemini-3.5-flash"
EMBED_MODEL = "all-MiniLM-L6-v2"

# ---------------------------------------------------------
# 0. ONTOLOGY & DRIVE MOUNT
# ---------------------------------------------------------
EQUIPMENT_HAZARDS = {
    "PUMP-14": ["Flammable Gas", "High Pressure", "Mechanical"],
    "COKE-OVEN-02": ["Thermal/Heat", "Toxic/Asphyxiant Gas", "Confined Space"],
    "BOILER-B1": ["High Pressure", "Thermal/Heat"],
    "VALVE-102": ["High Pressure", "Chemical Leak"],
    "CONVEYOR-C3": ["Mechanical", "Fire/Dust"],       # was "Mechanical/Moving Parts"
    "COMPRESSOR-A": ["High Pressure", "Mechanical"],
    "STORAGE-TANK-T5": ["Confined Space", "Flammable Gas"],
    "TRANSFORMER-TR1": ["Electrical", "Thermal/Heat"],
    "EXHAUST-FAN-F2": ["Mechanical"],                  # was "Mechanical/Moving Parts"
    "CRANE-OH-01": ["Mechanical"],                      # dropped "Suspended Load" — no reg covers it
}

REGULATORY_HAZARDS = {
    "factories_act_1948.pdf": ["Mechanical", "Thermal/Heat", "High Pressure", "Confined Space", "Chemical Leak"],
    "dgms_2020_04_gas_hazards.pdf": ["Flammable Gas", "Toxic/Asphyxiant Gas"],
    "dgms_2017_loto_procedures.pdf": ["Electrical", "Mechanical", "High Pressure"],
    "dgms_2020_05_oil_gas_safety.pdf": ["High Pressure", "Flammable Gas"]
}

KNOWN_EQUIPMENTS = list(EQUIPMENT_HAZARDS.keys())
KNOWN_HAZARDS = list(set([h for haz_list in EQUIPMENT_HAZARDS.values() for h in haz_list]))

def ensure_drive_mounted(mount_point="/content/drive"):
    if not os.path.ismount(mount_point):
        print("🔗 Mounting Google Drive...")
        drive.mount(mount_point)
    if not os.path.ismount(mount_point):
        raise RuntimeError(f"❌ Google Drive did NOT mount at {mount_point}.")
    print(f"✅ Google Drive verified as mounted at {mount_point}")

def verify_base_path(base_path):
    regulatory_dir = os.path.join(base_path, "regulatory")
    if not os.path.isdir(regulatory_dir):
        raise RuntimeError(f"❌ {regulatory_dir} does not exist.")
    pdfs = [f for f in os.listdir(regulatory_dir) if f.endswith(".pdf")]
    if not pdfs:
        raise RuntimeError(f"❌ {regulatory_dir} exists but has no PDFs in it.")
    print(f"✅ base_path verified — found {len(pdfs)} PDF(s) in {regulatory_dir}")

# ---------------------------------------------------------
# 1. VECTOR DB (PHASE 2)
# ---------------------------------------------------------
def hash_file(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        hasher.update(f.read())
    return hasher.hexdigest()

def ingest_regulatory_pdf(pdf_path, chunk_offset=0):
    print(f"⏳ Extracting {os.path.basename(pdf_path)} to Markdown...")
    md_text = pymupdf4llm.to_markdown(pdf_path)
    section_pattern = re.compile(r'(\n\*\*\d+[A-Z]?\.\s+[^\*]+\*\*)')
    parts = section_pattern.split(md_text)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
    chunks = []
    chunk_idx = chunk_offset

    if parts[0].strip():
        for split_txt in text_splitter.split_text(parts[0].strip()):
            chunks.append({
                "text": split_txt,
                "metadata": {"source": os.path.basename(pdf_path), "doc_type": "regulatory",
                             "section_title": "Preamble / TOC", "chunk_id": f"chunk_{chunk_idx}"}
            })
            chunk_idx += 1

    for i in range(1, len(parts), 2):
        header_raw, body_raw = parts[i].strip(), parts[i + 1].strip()
        clean_title = header_raw.replace('**', '').split('.—')[0].split('—')[0].strip()
        for split_txt in text_splitter.split_text(f"{header_raw}\n{body_raw}"):
            chunks.append({
                "text": split_txt,
                "metadata": {"source": os.path.basename(pdf_path), "doc_type": "regulatory",
                             "section_title": clean_title, "chunk_id": f"chunk_{chunk_idx}"}
            })
            chunk_idx += 1
    return chunks, chunk_idx

def ingest_synthetic_json(json_path):
    print(f"⏳ Extracting synthetic records from {os.path.basename(json_path)}...")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, dict): return []

    chunks = []
    for category in ['maintenance_logs', 'incident_reports', 'work_permits']:
        for item in data.get(category, []):
            record_id = (item.get('log_id') or item.get('incident_id') or item.get('permit_id') or hashlib.md5(str(item).encode()).hexdigest()[:8])
            text_lines = [f"Record Type: {category}"] + [f"{k}: {v}" for k, v in item.items()]
            chunks.append({
                "text": "\n".join(text_lines),
                "metadata": {
                    "source": os.path.basename(json_path), "doc_type": category,
                    "equipment_id": item.get("equipment_id", "UNKNOWN"), "record_id": record_id,
                    "chunk_id": f"{category}_{record_id}",
                }
            })
    return chunks

def build_vector_store(data_dirs, cache_path, chroma_path):
    manifest_path = cache_path.replace('.json', '_manifest.json')
    if os.path.exists(manifest_path) and os.path.exists(cache_path):
        print("📁 Loading existing text database from Drive cache...")
        with open(manifest_path, 'r', encoding='utf-8') as f: manifest = json.load(f)
        with open(cache_path, 'r', encoding='utf-8') as f: all_chunks = json.load(f)
    else:
        print("✨ No cache found. Starting fresh.")
        manifest, all_chunks = {}, []

    new_chunks = []
    chunk_counter = len(all_chunks)
    requires_db_update = False

    for d in data_dirs:
        if not os.path.exists(d): continue
        for f in sorted(os.listdir(d)):
            filepath = os.path.join(d, f)
            if (f.endswith('.pdf') or (f.endswith('.json') and 'synthetic' in d)) and os.path.getsize(filepath) > 0:
                current_hash = hash_file(filepath)
                if manifest.get(f) != current_hash:
                    print(f"🔄 Processing new/modified file: {f}")
                    requires_db_update, manifest[f] = True, current_hash
                    if f.endswith('.pdf'):
                        c, chunk_counter = ingest_regulatory_pdf(filepath, chunk_counter)
                        new_chunks.extend(c)
                    else:
                        new_chunks.extend(ingest_synthetic_json(filepath))
                else:
                    print(f"✅ Skipping unchanged file: {f}")

    local_emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
    chroma_client = chromadb.PersistentClient(path=chroma_path)
    collection = chroma_client.get_or_create_collection(name="industrial_brain_v2", embedding_function=local_emb_fn)

    if requires_db_update:
        print("💾 Writing new chunks + updated cache to Drive...")
        all_chunks.extend(new_chunks)
        with open(cache_path, 'w', encoding='utf-8') as f: json.dump(all_chunks, f, ensure_ascii=False, indent=4)
        with open(manifest_path, 'w', encoding='utf-8') as f: json.dump(manifest, f, ensure_ascii=False, indent=4)
        collection.upsert(
            documents=[d["text"] for d in new_chunks], metadatas=[d["metadata"] for d in new_chunks], ids=[d["metadata"]["chunk_id"] for d in new_chunks]
        )
        print(f"🚀 {len(new_chunks)} new chunks added.")
    elif collection.count() == 0 and len(all_chunks) > 0:
        print("💾 Re-hydrating vector store from cache...")
        collection.upsert(
            documents=[d["text"] for d in all_chunks], metadatas=[d["metadata"] for d in all_chunks], ids=[d["metadata"]["chunk_id"] for d in all_chunks]
        )
    print(f"📊 Total chunks in Vector DB: {collection.count()}")
    return collection

# ---------------------------------------------------------
# 2. KNOWLEDGE GRAPH (PHASE 3)
# ---------------------------------------------------------
def build_knowledge_graph(json_path, graph_path):
    manifest_path = graph_path.replace('.pkl', '_manifest.json')
    current_hash = hash_file(json_path) if os.path.exists(json_path) else None

    if os.path.exists(graph_path) and os.path.exists(manifest_path):
        with open(manifest_path, 'r') as f: manifest = json.load(f)
        if manifest.get('industrial_records.json') == current_hash:
            print("📁 Loading existing Knowledge Graph from Drive cache...")
            with open(graph_path, 'rb') as f: return pickle.load(f)

    print("🕸️ Building new Knowledge Graph...")
    G = nx.MultiDiGraph()

    for eq, hazards in EQUIPMENT_HAZARDS.items():
        G.add_node(eq, type="Equipment")
        for haz in hazards:
            G.add_node(haz, type="HazardType")
            G.add_edge(eq, haz, label="HAS_HAZARD")

    for reg, hazards in REGULATORY_HAZARDS.items():
        G.add_node(reg, type="RegulationClause")
        for haz in hazards:
            G.add_node(haz, type="HazardType")
            G.add_edge(reg, haz, label="APPLIES_TO")

    G.add_node("oisd-gdn-207.pdf", type="RegulationClause")

    with open(json_path, 'r', encoding='utf-8') as f: data = json.load(f)

    unique_plant_sections = set()
    equipment_locations = {}

    for item in data.get('work_permits', []):
        permit_id = item.get('permit_id') or hashlib.md5(str(item).encode()).hexdigest()[:8]
        eq_id = item.get('equipment_id', 'UNKNOWN')
        approver = item.get('approving_officer', 'UNKNOWN')
        location = item.get('location', 'UNKNOWN')

        unique_plant_sections.add(location)
        if eq_id != 'UNKNOWN': equipment_locations[eq_id] = location

        G.add_node(permit_id, type="Permit", **item)
        G.add_node(approver, type="PersonRole")
        G.add_node(location, type="PlantSection")
        G.add_edge(permit_id, eq_id, label="ISSUED_FOR")
        G.add_edge(permit_id, approver, label="APPROVED_BY")

    for item in data.get('maintenance_logs', []):
        log_id = item.get('log_id') or hashlib.md5(str(item).encode()).hexdigest()[:8]
        eq_id = item.get('equipment_id', 'UNKNOWN')
        G.add_node(log_id, type="MaintenanceLog", **item)
        G.add_edge(log_id, eq_id, label="MAINTAINED")

    for item in data.get('incident_reports', []):
        inc_id = item.get('incident_id') or hashlib.md5(str(item).encode()).hexdigest()[:8]
        eq_id = item.get('equipment_id', 'UNKNOWN')
        G.add_node(inc_id, type="Incident", **item)
        G.add_edge(inc_id, eq_id, label="INVOLVED")

    for eq_id, loc in equipment_locations.items():
        G.add_edge(eq_id, loc, label="LOCATED_IN")

    for section in unique_plant_sections:
        G.add_edge("oisd-gdn-207.pdf", section, label="APPLIES_TO_ALL")

    print("💾 Saving Knowledge Graph to Drive...")
    with open(graph_path, 'wb') as f: pickle.dump(G, f)
    with open(manifest_path, 'w') as f: json.dump({'industrial_records.json': current_hash}, f)

    print("\n📊 --- KNOWLEDGE GRAPH METRICS ---")
    node_types, edge_types = {}, {}
    for _, d in G.nodes(data=True):
        ntype = d.get('type', 'Unknown')
        node_types[ntype] = node_types.get(ntype, 0) + 1
    for nt, count in node_types.items(): print(f"  Nodes [{nt}]: {count}")
    for _, _, d in G.edges(data=True):
        etype = d.get('label', 'Unknown')
        edge_types[etype] = edge_types.get(etype, 0) + 1
    for et, count in edge_types.items(): print(f"  Edges [{et}]: {count}")
    return G

# ---------------------------------------------------------
# 3. RETRIEVAL & HYBRID REASONING
# ---------------------------------------------------------
def expand_query(query, doc_type=None):
    if doc_type == "regulatory":
        instruction = "Expand this query into related legal/regulatory terms and section references for Indian industrial safety law."
    elif doc_type is None:
        instruction = "Expand this query into terms covering BOTH regulatory/legal language AND operational terms."
    else:
        instruction = "Expand this query into related equipment, maintenance, and operational terms. No legal jargon."
    response = completion(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": f"{instruction}\n\nQuery: {query}"}],
        temperature=0.0, api_key=userdata.get('GEMINI_API_KEY')
    )
    return f"{query} {response.choices[0].message.content.strip()}"

def retrieve_chunks(collection, expanded_query, doc_type=None, n_results=5):
    results = collection.query(
        query_texts=[expanded_query], n_results=n_results,
        where={"doc_type": doc_type} if doc_type else None
    )
    ids = results['ids'][0]
    formatted = [f"--- ID: {cid} (Source: {m['source']}) ---\n{txt}" for cid, m, txt in zip(ids, results['metadatas'][0], results['documents'][0])]
    return "\n\n".join(formatted), ids

def query_knowledge_graph(G, entity_id, relation=None):
    results = []
    if entity_id not in G: return results
    for u, v, data in G.out_edges(entity_id, data=True):
        if relation is None or data.get('label') == relation: results.append((entity_id, data.get('label'), v))
    for u, v, data in G.in_edges(entity_id, data=True):
        if relation is None or data.get('label') == relation: results.append((u, data.get('label'), entity_id))
    return results

def get_regulations_for_equipment(G, equipment_id):
    applicable_regs = set()
    if equipment_id not in G: return applicable_regs
    hazards = [v for u, v, d in G.out_edges(equipment_id, data=True) if d.get('label') == 'HAS_HAZARD']
    for haz in hazards:
        for u, v, d in G.in_edges(haz, data=True):
            if d.get('label') == 'APPLIES_TO': applicable_regs.add((u, haz))
    locations = [v for u, v, d in G.out_edges(equipment_id, data=True) if d.get('label') == 'LOCATED_IN']
    for loc in locations:
        for u, v, d in G.in_edges(loc, data=True):
            if d.get('label') == 'APPLIES_TO_ALL': applicable_regs.add((u, loc))
    return applicable_regs

def extract_graph_context(query, G):
    graph_context = []
    for eq in KNOWN_EQUIPMENTS:
        if eq.lower() in query.lower():
            for u, rel, v in query_knowledge_graph(G, eq):
                graph_context.append(f"GRAPH FACT: [{u}] -> {rel} -> [{v}] [graph:{rel}]")
            for reg, target in get_regulations_for_equipment(G, eq):
                if target in KNOWN_HAZARDS:
                    graph_context.append(f"GRAPH FACT: [{reg}] applies to hazard [{target}] which is present in [{eq}] [graph:APPLIES_TO]")
                else:
                    graph_context.append(f"GRAPH FACT: [{reg}] applies broadly to plant section [{target}] where [{eq}] is located [graph:APPLIES_TO_ALL]")
    for haz in KNOWN_HAZARDS:
        if haz.lower() in query.lower():
            equipments = [u for u, v, d in G.in_edges(haz, data=True) if d.get('label') == 'HAS_HAZARD']
            for eq in equipments:
                graph_context.append(f"GRAPH FACT: Equipment [{eq}] has hazard [{haz}] [graph:HAS_HAZARD]")
                incidents = [u for u, v, d in G.in_edges(eq, data=True) if d.get('label') == 'INVOLVED']
                for inc in incidents:
                    graph_context.append(f"GRAPH FACT: Incident [{inc}] INVOLVED Equipment [{eq}] [graph:INVOLVED]")
    return list(set(graph_context))

def verify_citations(answer, context_str):
    """LLM-as-a-judge for BOTH Phase 2 Vector and Phase 3 Graph citations."""
    prompt = f"""Check whether every bracketed citation (e.g., [chunk_12] or [graph:ISSUED_FOR]) in ANSWER is genuinely supported by its matching source in CONTEXT.
A citation naming a source that only lists a section number or node, without stating the actual fact claimed, counts as unsupported.

CONTEXT:
{context_str}

ANSWER:
{answer}

Respond in strict JSON only, no markdown fences:
{{"grounded": true/false, "unsupported_claims": ["..."]}}"""

    response = completion(
        model=MODEL_NAME, messages=[{"role": "user", "content": prompt}],
        temperature=0.0, api_key=userdata.get('GEMINI_API_KEY')
    )
    raw = re.sub(r'^```json\s*|\s*```$', '', response.choices[0].message.content.strip())
    try: return json.loads(raw)
    except json.JSONDecodeError: return {"grounded": None, "unsupported_claims": [f"verifier returned non-JSON: {raw[:200]}"]}

# Cache Helpers
def _query_cache_key(query, doc_type): return hashlib.md5(f"{query}::{doc_type}".encode()).hexdigest()
def load_query_cache(path): return json.load(open(path, 'r', encoding='utf-8')) if os.path.exists(path) else {}
def save_query_cache(path, cache): json.dump(cache, open(path, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

def ask(collection, query, query_cache_path, doc_type=None, force_refresh=False):
    """Standard Vector-only search from Phase 2."""
    cache = load_query_cache(query_cache_path)
    key = _query_cache_key(query, doc_type)
    if not force_refresh and key in cache: return cache[key]

    expanded = expand_query(query, doc_type)
    context, ids = retrieve_chunks(collection, expanded, doc_type)

    if not len(ids):
        result = {"query": query, "answer": "No relevant info found.", "grounded": True, "unsupported_claims": [], "chunks_used": []}
    else:
        answer = completion(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Synthesize an answer using ONLY the provided context. Cite the source chunk ID in [brackets] after every factual claim. If info is missing, say 'INFO NOT IN CONTEXT'."},
                {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUERY: {query}"}
            ],
            temperature=0.0, api_key=userdata.get('GEMINI_API_KEY')
        ).choices[0].message.content

        verification = verify_citations(answer, context)
        result = {"query": query, "answer": answer, "grounded": verification.get("grounded"), "unsupported_claims": verification.get("unsupported_claims", []), "chunks_used": list(ids)}

    print(f"\n{'='*60}\nQ: {result['query']}\n{'='*60}\nA: {result['answer']}")
    print(f"Grounded: {result['grounded']} | Sources: {result['chunks_used']}")
    cache[key] = result
    save_query_cache(query_cache_path, cache)
    return result

def ask_hybrid(collection, G, query, query_cache_path, doc_type=None, force_refresh=False):
    """Phase 3 Hybrid search combining Vectors and Knowledge Graph."""
    cache = load_query_cache(query_cache_path)
    key = _query_cache_key(f"hybrid::{query}", doc_type)

    if not force_refresh and key in cache:
        print(f"\n{'='*70}\n🧠 HYBRID QUERY: {query}  [CACHED]\n{'='*70}")
        print(f"A: {cache[key]['answer']}")
        print(f"Grounded: {cache[key].get('grounded')} | Sources -> Vectors: {len(cache[key].get('chunks_used', []))} | Graph: {len(cache[key].get('graph_facts_used', []))}")
        return cache[key]['answer']

    print(f"\n{'='*70}\n🧠 HYBRID QUERY: {query}\n{'='*70}")

    graph_facts = extract_graph_context(query, G)
    graph_context_str = "\n".join(graph_facts)
    if graph_facts: print(f"🕸️ Extracted {len(graph_facts)} Knowledge Graph relationships.")

    expanded = expand_query(query, doc_type)
    vector_context_str, ids = retrieve_chunks(collection, expanded, doc_type)

    if not len(ids) and not graph_facts: return "No relevant info found."

    full_context = "--- KNOWLEDGE GRAPH FACTS ---\n" + (graph_context_str if graph_context_str else "None") + "\n\n--- VECTOR DATABASE CHUNKS ---\n" + vector_context_str

    system_prompt = (
        "You are an Industrial Intelligence AI. Synthesize a flawless answer using ONLY the provided context.\n"
        "RULES FOR CITATIONS:\n"
        "1. If you use a vector database chunk, cite it using its ID exactly, e.g., [chunk_12] or [work_permits_WP-302].\n"
        "2. If you use a Knowledge Graph fact, cite it using its relation exactly, e.g., [graph:ISSUED_FOR] or [graph:APPLIES_TO].\n"
        "3. You MUST distinguish between these two sources. Every claim must have a citation.\n"
        "4. If the exact answer is missing, output 'INFO NOT IN CONTEXT'."
    )

    answer = completion(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": f"CONTEXT:\n{full_context}\n\nQUERY: {query}"}],
        temperature=0.0, api_key=userdata.get('GEMINI_API_KEY')
    ).choices[0].message.content

    verification = verify_citations(answer, full_context)

    print(f"A: {answer}")
    print(f"\nGrounded: {verification.get('grounded')} | Sources Used -> Vectors: {len(ids)} | Graph: {len(graph_facts)}")
    if verification.get("unsupported_claims"): print(f"⚠️  Unsupported: {verification.get('unsupported_claims')}")

    cache[key] = {
        "query": query, "answer": answer, "chunks_used": list(ids),
        "graph_facts_used": graph_facts, "grounded": verification.get("grounded"),
        "unsupported_claims": verification.get("unsupported_claims", [])
    }
    save_query_cache(query_cache_path, cache)
    return answer

# ---------------------------------------------------------
# 4. EXECUTION PIPELINE
# ---------------------------------------------------------
if __name__ == "__main__":
    # Setup
    ensure_drive_mounted()
    base_path = "/content/drive/MyDrive/ET_Industrial_data"
    verify_base_path(base_path)

    data_dirs = [os.path.join(base_path, "regulatory"), os.path.join(base_path, "synthetic")]
    cache_path = os.path.join(base_path, "master_chunks.json")
    chroma_path = os.path.join(base_path, "vectorstore")
    json_path = os.path.join(base_path, "synthetic", "industrial_records.json")
    graph_path = os.path.join(base_path, "knowledge_graph.pkl")
    query_cache_path = os.path.join(base_path, "query_cache.json")
    os.makedirs(chroma_path, exist_ok=True)

    # Initialize Databases
    print(f"\n🚀 Initializing Industrial Brain at: {base_path}")
    db_collection = build_vector_store(data_dirs, cache_path, chroma_path)
    KG = build_knowledge_graph(json_path, graph_path)

    # Run Tests
    print("\n--- TEST A: Graph Reasoning for 'PUMP-14' ---")
    regs = get_regulations_for_equipment(KG, "PUMP-14")
    for r, h in regs: print(f"Regulation: {r}  -->  Covers Hazard/Location: {h}")

    print("\n--- TEST B: Hybrid Ask (Permits + Graph Derived Regulations) ---")
    ask_hybrid(db_collection, KG, "What permits were active near PUMP-14, and does the Factory Act apply to it?", query_cache_path, force_refresh=True)

    import time
    time.sleep(4) # Rate limit breather

    print("\n--- TEST C: Multi-hop reasoning (Hazard -> Equipment -> Incident) ---")
    ask_hybrid(db_collection, KG, "What incidents involved equipment with Confined Space hazard exposure?", query_cache_path, force_refresh=True)

✅ Google Drive verified as mounted at /content/drive
✅ base_path verified — found 5 PDF(s) in /content/drive/MyDrive/ET_Industrial_data/regulatory

🚀 Initializing Industrial Brain at: /content/drive/MyDrive/ET_Industrial_data
📁 Loading existing text database from Drive cache...
✅ Skipping unchanged file: dgms_2017_loto_procedures.pdf
✅ Skipping unchanged file: dgms_2020_04_gas_hazards.pdf
✅ Skipping unchanged file: dgms_2020_05_oil_gas_safety.pdf
✅ Skipping unchanged file: factories_act_1948.pdf
✅ Skipping unchanged file: oisd-gdn-207.pdf
✅ Skipping unchanged file: industrial_records.json


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


📊 Total chunks in Vector DB: 513
📁 Loading existing Knowledge Graph from Drive cache...

--- TEST A: Graph Reasoning for 'PUMP-14' ---
Regulation: dgms_2020_04_gas_hazards.pdf  -->  Covers Hazard/Location: Flammable Gas
Regulation: dgms_2017_loto_procedures.pdf  -->  Covers Hazard/Location: High Pressure
Regulation: dgms_2017_loto_procedures.pdf  -->  Covers Hazard/Location: Mechanical
Regulation: factories_act_1948.pdf  -->  Covers Hazard/Location: High Pressure
Regulation: dgms_2020_05_oil_gas_safety.pdf  -->  Covers Hazard/Location: Flammable Gas
Regulation: dgms_2020_05_oil_gas_safety.pdf  -->  Covers Hazard/Location: High Pressure
Regulation: factories_act_1948.pdf  -->  Covers Hazard/Location: Mechanical
Regulation: oisd-gdn-207.pdf  -->  Covers Hazard/Location: Gas Plant Area

--- TEST B: Hybrid Ask (Permits + Graph Derived Regulations) ---

🧠 HYBRID QUERY: What permits were active near PUMP-14, and does the Factory Act apply to it?
🕸️ Extracted 19 Knowledge Graph relationships.

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


A: Based on the provided context, here are the details regarding the permits active for PUMP-14 and the applicability of the Factories Act:

### Active Permits for PUMP-14
Two work permits were issued for PUMP-14 [graph:ISSUED_FOR]:
1. **WP-301**: A **Hot Work** permit for PUMP-14 located in the Gas Plant Area. It was active on **2026-06-10** and approved by the Safety Officer [work_permits_WP-301].
2. **WP-308**: A **Cold Work** permit for PUMP-14 located in the Gas Plant Area. It was active on **2026-06-28** and approved by the Shift Supervisor [work_permits_WP-308].

### Applicability of the Factories Act
Yes, the Factories Act (`factories_act_1948.pdf`) applies to PUMP-14 [graph:APPLIES_TO]. Specifically, the Act applies to the following hazards present in PUMP-14:
* **High Pressure** [graph:APPLIES_TO, graph:HAS_HAZARD]
* **Mechanical** [graph:APPLIES_TO, graph:HAS_HAZARD]

Grounded: True | Sources Used -> Vectors: 5 | Graph: 19

--- TEST C: Multi-hop reasoning (Hazard -> Equipmen

In [3]:
print(get_regulations_for_equipment(KG, "CONVEYOR-C3"))

{('dgms_2017_loto_procedures.pdf', 'Mechanical'), ('oisd-gdn-207.pdf', 'Raw Material Handling'), ('factories_act_1948.pdf', 'Mechanical')}
